# Customer Churn Prediction — End-to-End ML Mini Project
**Flow:** Data Understanding → Cleaning → EDA → Correlation → Feature Engineering → Feature Selection → Train/Test Split → Models → Evaluation → Tuning → Final Model → Save

**Target:** `Churned` (yes = customer cancelled, no = customer stayed)
> Note: the assignment says "chunk" – it means **churn**.

Put `Customer_Cancellations_Training.csv` and `preprocessing.py` in the same folder as this notebook.

## 1. Data Understanding

**Segment 1.1 – Import libraries.** We import everything once at the top so the rest of the notebook stays clean.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score, roc_curve)

from preprocessing import add_features, build_pipeline

sns.set_style("whitegrid")
RANDOM_STATE = 42

**Segment 1.2 – Load the dataset and look at rows/columns.** `shape` gives (rows, columns). `head()` shows the first 5 rows so we see what the data looks like.

In [ ]:
df = pd.read_csv("Customer_Cancellations_Training.csv")
print("Rows, Columns:", df.shape)
df.head()

**Segment 1.3 – Data types and null counts.** `info()` tells us each column's type. `object/str` = text (categorical), `int/float` = numbers.

In [ ]:
df.info()

**Segment 1.4 – Separate numerical and categorical columns, identify the target.** We keep the target out of the feature lists.

In [ ]:
target = "Churned"
numerical_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in df.select_dtypes(exclude="number").columns if c != target]

print("Target             :", target)
print("Numerical columns  :", numerical_cols)
print("Categorical columns:", categorical_cols)

**Segment 1.5 – Descriptive statistics.** `describe()` gives mean, std, min, quartiles, max for numbers – good for spotting strange values. `include='object'` does the same for text columns (unique values, most frequent value).

In [ ]:
df[numerical_cols].describe().T

In [ ]:
df[categorical_cols + [target]].describe(include='all').T

**Segment 1.6 – Class distribution of the target.** If one class is much bigger than the other the data is *imbalanced*, and accuracy alone becomes misleading.

In [ ]:
print(df[target].value_counts())
print()
print((df[target].value_counts(normalize=True) * 100).round(2))

**Observation:** about 85.6% of customers stayed and only 14.4% churned → the data is **imbalanced**.
A model that always says "no churn" would already be ~85.6% accurate but useless. So we will focus on **Recall, F1 and ROC-AUC**, not accuracy alone.

## 2. Data Cleaning

**Segment 2.1 – Missing values.** Models cannot learn from NaN. We count nulls per column.

In [ ]:
df.isnull().sum()

**Observation:** no missing values. But hidden missing values can be written as symbols like `-`, so we check that in 2.5.

**Segment 2.2 – Duplicates.** Duplicate rows give the model the same example twice (can inflate scores). We check full-row duplicates and also duplicates ignoring `AccountID` (an ID is unique by nature).

In [ ]:
print("Full duplicate rows        :", df.duplicated().sum())
print("Duplicates ignoring AccountID:", df.drop(columns="AccountID").duplicated().sum())
print("Duplicate AccountIDs       :", df["AccountID"].duplicated().sum())

**Observation:** no duplicates, so nothing to remove.

**Segment 2.3 – Data-type handling.** The target is text (`yes`/`no`). ML models need numbers, so we convert it to 1/0. We make a copy so the original `df` stays untouched.

In [ ]:
data = df.copy()
data[target] = data[target].map({"yes": 1, "no": 0})
print(data[target].value_counts())
print(data.dtypes)

**Segment 2.4 – Inconsistent categorical values.** We strip spaces, and check the unique values of each text column for spelling/case differences (e.g. `Yes`, `yes `, `YES`).

In [ ]:
for col in ["DeviceType", "Promotion", "HasRenewed", "PlanType", "StartMonth", "Country"]:
    data[col] = data[col].str.strip()
    print(col, "->", data[col].unique())

**Observation:** values are consistent (no mixed case/spelling problems).

**Segment 2.5 – Invalid values.** We check for negative numbers, impossible values and placeholder symbols.

In [ ]:
print("Negative values per numeric column:")
print((data[numerical_cols] < 0).sum())
print()
print("Rows with PriorPeriodUsage == 0 :", (data["PriorPeriodUsage"] == 0).sum())
print("Rows with ServiceRating  == 0   :", (data["ServiceRating"] == 0).sum())
print("DeviceType '-' count            :", (data["DeviceType"] == "-").sum())

**Observation / decision:**
- No negative values.
- `DeviceType` has `-` for 2,160 rows (68%). It is a placeholder meaning *device not known*. We rename it to **`Unknown`** (done inside `add_features` so the API does the same).
- One row has `PriorPeriodUsage = 0` – possible for a brand-new usage period, so we keep it but make ratio features safe from divide-by-zero.
- 16 rows have `ServiceRating = 0`. The scale runs 0–20, so 0 is a valid (very low) rating and we keep it.

In [ ]:
data["DeviceType"] = data["DeviceType"].replace("-", "Unknown")
data["DeviceType"].value_counts()

**Segment 2.6 – Outlier detection using Z-score.**
Z-score = (value − mean) / std. A value with |z| > 3 is more than 3 standard deviations from the mean → treated as an outlier.
We *detect* first (count per column).

In [ ]:
outlier_cols = ["BaseFee", "AdditionalFeatureSpend", "NumberOfPenalties", "CurrentPeriodUsage",
                "PriorPeriodUsage", "ServiceRating", "ServiceTickets", "CustomerTenure"]

z_scores = np.abs(zscore(data[outlier_cols]))
outlier_count = (z_scores > 3).sum(axis=0)
pd.Series(outlier_count, index=outlier_cols, name="outliers (|z|>3)")

**Segment 2.7 – Handle outliers by capping (winsorizing).**
Customer data is small (3,160 rows) so deleting rows loses information. Instead we **cap** every value to `mean ± 3·std`.
These are real customers with unusual behaviour, not typing errors, so capping keeps them but reduces their pull on the model.

In [ ]:
for col in outlier_cols:
    mean, std = data[col].mean(), data[col].std()
    lower, upper = mean - 3 * std, mean + 3 * std
    data[col] = data[col].clip(lower, upper)

# confirm - nothing should be beyond |z|>3 by much now
print((np.abs(zscore(data[outlier_cols])) > 3).sum(axis=0))

**Segment 2.8 – Unnecessary columns.**
| Column | Why drop |
|---|---|
| `AccountID` | Unique ID for each row – no pattern to learn, causes overfitting |
| `Country` | Only one value (`USA`) – zero information |
| `StartWeek` | 53 values and fully determined by `StartMonth` (redundant, high cardinality) |

In [ ]:
data = data.drop(columns=["AccountID", "Country", "StartWeek"])
print(data.shape)
data.head()

**Segment 2.9 – Leakage detection.**
*Data leakage* = a column that contains information from the future / from the answer itself, which would not be available when predicting.
Suspicious column: **`HasRenewed`**. We check how strongly it relates to the target and how the model score changes with and without it.

In [ ]:
print(pd.crosstab(data["HasRenewed"], data[target], normalize="index").round(3))

# quick check: score of a simple model with and without HasRenewed
tmp = data.drop(columns=target)
tmp_y = data[target]
tmp = add_features(tmp)
tmp_num = tmp.select_dtypes(include="number").columns.tolist()

def quick_auc(cols):
    X_tmp = pd.get_dummies(tmp[cols], drop_first=True)
    return cross_val_score(RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
                           X_tmp, tmp_y, cv=5, scoring="roc_auc").mean()

base_cols = tmp_num + ["PlanType", "Promotion"]
print("ROC-AUC without HasRenewed:", round(quick_auc(base_cols), 4))
print("ROC-AUC with    HasRenewed:", round(quick_auc(base_cols + ["HasRenewed"]), 4))

**Observation / decision:** `HasRenewed = yes` has a *higher* churn rate (~42%) than `no` (~11.5%), which is unusual. It is not a perfect predictor (AUC only moves from about 0.928 to 0.953), so it is **not a hard leakage**.
But we flag it: if `HasRenewed` is only known *after* the customer decides, it should not be used. We **keep it** (it is a normal input column of the dataset) and mention this in the viva as a business question to confirm.

## 3. Exploratory Data Analysis (EDA)

**Segment 3.1 – Target class distribution.** Count plot with a different colour per class.

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x=data[target].map({0: "No churn", 1: "Churn"}), palette=["#4C9F70", "#D1495B"])
plt.title("Target class distribution")
plt.xlabel("")
plt.show()

**Observation:** strong class imbalance (about 6 : 1).

**Segment 3.2 – Histograms (numerical distributions).** Shows the shape of each feature: normal, skewed, etc. `kde=True` draws a smooth curve on top.

In [ ]:
eda_num = ["BaseFee", "AdditionalFeatureSpend", "NumberOfPenalties", "CurrentPeriodUsage",
           "PriorPeriodUsage", "ServiceRating", "ServiceTickets", "CustomerTenure"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), eda_num):
    sns.histplot(data[col], kde=True, ax=ax, color="#3A6EA5")
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Observation:** usage, fee, rating and tenure are roughly bell-shaped; penalties and tickets are right-skewed counts (most customers have few).

**Segment 3.3 – Box plots.** The box shows the middle 50% of values; dots beyond the whiskers are potential outliers (after capping, far fewer remain).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), eda_num):
    sns.boxplot(y=data[col], ax=ax, color="#8FBC8F")
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Segment 3.4 – Violin plots: numeric feature vs target.** A violin is a box plot + a density shape, so we see *where most values sit* for churned and non-churned customers.

In [ ]:
plot_df = data.copy()
plot_df["Churn"] = plot_df[target].map({0: "No", 1: "Yes"})

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), eda_num):
    sns.violinplot(data=plot_df, x="Churn", y=col, palette=["#4C9F70", "#D1495B"], ax=ax)
    ax.set_title(f"{col} vs Churn")
plt.tight_layout()
plt.show()

**Observation:** churned customers tend to have a **higher BaseFee** and **more penalties**. Tenure, rating and tickets look almost the same for both groups.

**Segment 3.5 – Count plots for categorical features (multi-colour).** `hue` splits every bar by churn, so we compare class counts inside each category.

In [ ]:
cat_plot_cols = ["DeviceType", "Promotion", "HasRenewed", "PlanType"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), cat_plot_cols):
    sns.countplot(data=plot_df, x=col, hue="Churn", palette=["#4C9F70", "#D1495B"], ax=ax)
    ax.set_title(f"{col} (count by churn)")
plt.tight_layout()
plt.show()

**Segment 3.6 – Churn rate per category (feature vs target).** Raw counts can mislead when categories have different sizes, so we plot the **percentage** that churned.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, col in zip(axes.ravel(), ["DeviceType", "Promotion", "HasRenewed", "PlanType", "StartMonth"]):
    rate = data.groupby(col)[target].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.index, y=rate.values, palette="viridis", ax=ax)
    ax.set_title(f"Churn rate (%) by {col}")
    ax.set_ylabel("Churn %")
    ax.tick_params(axis="x", rotation=45)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

**Observation:**
- **PlanType is the strongest driver:** Red Plan customers churn ~67% vs only 3–13% for the others.
- Customers **without a promotion** churn more (16.6% vs 8.7%).
- `HasRenewed = yes` shows higher churn (see leakage note).
- DeviceType and StartMonth show almost no difference → weak features.

**Segment 3.7 – Outlier analysis visual.** Box plot of `CurrentPeriodUsage` and `ServiceRating` by churn, after capping.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=plot_df, x="Churn", y="CurrentPeriodUsage", palette=["#4C9F70", "#D1495B"], ax=axes[0])
sns.boxplot(data=plot_df, x="Churn", y="ServiceRating", palette=["#4C9F70", "#D1495B"], ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

**Segment 4.1 – Correlation heatmap.** Correlation is between −1 and +1. Near ±1 = strong relationship, near 0 = none. We include the target so we also see feature-vs-target correlation.

In [ ]:
corr = data[eda_num + [target]].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation heatmap")
plt.show()

**Segment 4.2 – List highly correlated pairs** (|corr| > 0.7) to detect multicollinearity.

In [ ]:
corr_features = data[eda_num].corr().abs()
pairs = (corr_features.where(np.triu(np.ones(corr_features.shape), k=1).astype(bool))
         .stack().sort_values(ascending=False))
pairs[pairs > 0.7]

In [ ]:
print("Correlation of each feature with the target:")
corr[target].drop(target).sort_values()

**Observation:**
- **Highly correlated:** `AdditionalFeatureSpend` ↔ `PriorPeriodUsage` ≈ **0.998** → almost the same information (spend ≈ 11.8 × usage). This is **multicollinearity**: it makes Logistic Regression coefficients unstable and wastes model capacity.
- **Negatively correlated with churn:** `ServiceTickets` (weak, ≈ −0.05).
- **Positively correlated with churn:** `BaseFee` (≈ 0.21), `NumberOfPenalties` (≈ 0.20).
- **Almost no correlation:** `CustomerTenure`, `CurrentPeriodUsage`, `ServiceRating`.
- **Decision:** drop `AdditionalFeatureSpend` (keep `PriorPeriodUsage`).

In [ ]:
data = data.drop(columns=["AdditionalFeatureSpend"])
data.columns.tolist()

## 5. Feature Engineering

**Segment 5.1 – Create new features.** The function `add_features` lives in `preprocessing.py` (so the API can reuse the *same* code). It builds:

| New feature | Formula | Idea |
|---|---|---|
| `UsageChange` | Current − Prior usage | is the customer using the service less now? |
| `UsageChangePct` | UsageChange / (Prior+1) | relative change (+1 avoids divide by zero) |
| `FeePerUsage` | BaseFee / CurrentUsage | price paid for what they use |
| `PenaltiesPerTenure` | Penalties / (Tenure+1) | penalty *rate* not just count |
| `TicketsPerTenure` | Tickets / (Tenure+1) | complaint rate |
| `PenaltyFeeInteraction` | Penalties × BaseFee | interaction feature |
| `TenureGroup` | bins of tenure | New / Growing / Established / Loyal |
| `HasDeviceInfo` | DeviceType != Unknown | 0/1 flag |
| `StartQuarter` | from StartMonth | Q1–Q4 (date-based feature) |

In [ ]:
X = data.drop(columns=target)
y = data[target]

X_fe = add_features(X)
print("Columns before:", X.shape[1], "| after:", X_fe.shape[1])
X_fe.head()

**Segment 5.2 – Do the new features help?** We check correlation of the new numeric features with the target.

In [ ]:
new_num = ["UsageChange", "UsageChangePct", "FeePerUsage", "PenaltiesPerTenure",
           "TicketsPerTenure", "PenaltyFeeInteraction", "HasDeviceInfo"]
X_fe[new_num].assign(Churned=y.values).corr()["Churned"].drop("Churned").sort_values()

**Segment 5.3 – Encoding & scaling (explained).** We do **not** transform the data by hand here. Instead `build_pipeline()` in `preprocessing.py` puts these steps *inside the model pipeline*:
- `StandardScaler` for numeric columns → mean 0, std 1 (needed by Logistic Regression; harmless for trees).
- `OneHotEncoder` for categorical columns → turns `PlanType` into `Plan_Red`, `Plan_Blue`… 0/1 columns. `handle_unknown="ignore"` stops the API from crashing on an unseen category.

Why inside the pipeline? Scaler/encoder must **learn only from the training data** (to avoid leakage), and the exact same steps must run again at prediction time.

## 6. Feature Selection

**Segment 6.1 – Mutual Information.** Measures how much knowing a feature reduces uncertainty about churn (works for non-linear relations too; 0 = useless).

In [ ]:
all_num = ["BaseFee", "NumberOfPenalties", "CurrentPeriodUsage", "PriorPeriodUsage", "ServiceRating",
           "ServiceTickets", "CustomerTenure", "UsageChange", "UsageChangePct", "FeePerUsage",
           "PenaltiesPerTenure", "TicketsPerTenure", "PenaltyFeeInteraction", "HasDeviceInfo"]
all_cat = ["Territory", "DeviceType", "Promotion", "HasRenewed", "PlanType", "StartMonth",
           "StartQuarter", "TenureGroup"]

X_enc = pd.get_dummies(X_fe[all_num + all_cat], drop_first=False)
mi = mutual_info_classif(X_enc, y, random_state=RANDOM_STATE)
mi_series = pd.Series(mi, index=X_enc.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 7))
mi_series.head(20).sort_values().plot(kind="barh", color="#3A6EA5")
plt.title("Top 20 features by Mutual Information")
plt.show()

**Segment 6.2 – Random Forest feature importance (model-based).** A forest counts how much each feature reduces impurity across all trees.

In [ ]:
rf_fs = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                               random_state=RANDOM_STATE, n_jobs=-1)
rf_fs.fit(X_enc, y)

imp = pd.Series(rf_fs.feature_importances_, index=X_enc.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 7))
imp.head(20).sort_values().plot(kind="barh", color="#D1495B")
plt.title("Top 20 features by Random Forest importance")
plt.show()

**Observation:**
- Strong: `PlanType_Red Plan`, `BaseFee`, `NumberOfPenalties`, `PenaltyFeeInteraction`, `PenaltiesPerTenure`, `HasRenewed`, usage features.
- `Promotion` ranks lower in importance but its churn difference is statistically significant (16.6% vs 8.7%), so we keep it.
- Weak: `Territory` (51 states → 51 tiny sparse columns that mostly add noise), `StartMonth`, `StartQuarter`, `DeviceType`, `TenureGroup`, `CustomerTenure`.

**Segment 6.3 – Compare performance BEFORE and AFTER feature selection.**
We use 5-fold stratified cross-validation (on the data, with the full pipeline) and compare ROC-AUC and F1 for all features vs the selected features.

In [ ]:
selected_num = ["BaseFee", "NumberOfPenalties", "CurrentPeriodUsage", "PriorPeriodUsage",
                "ServiceRating", "ServiceTickets", "UsageChange", "PenaltyFeeInteraction"]
selected_cat = ["Promotion", "HasRenewed", "PlanType"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def cv_scores(num_cols, cat_cols, name):
    pipe = build_pipeline(num_cols, cat_cols,
                          RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                 random_state=RANDOM_STATE, n_jobs=-1))
    auc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()
    f1 = cross_val_score(pipe, X, y, cv=cv, scoring="f1").mean()
    return {"Feature set": name, "n_features": len(num_cols) + len(cat_cols), "ROC-AUC": round(auc, 4), "F1": round(f1, 4)}

fs_result = pd.DataFrame([
    cv_scores(all_num, all_cat, "All features"),
    cv_scores(selected_num, selected_cat, "Selected features"),
])
fs_result

**Observation:** the selected (smaller) feature set performs **as well or better** than using everything. Removing noisy columns (Territory, month, device, tenure bins) reduces overfitting and makes the model simpler and faster.
**Decision:** use `selected_num` and `selected_cat` from now on.

## 7. Train / Test Split

**Segment 7.1 – Split into train (80%) and test (20%).**
`stratify=y` keeps the same churn percentage (~14%) in both parts – important for imbalanced data. `random_state` makes the split reproducible.
The test set is kept untouched until final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn % in train:", round(y_train.mean() * 100, 2))
print("Churn % in test :", round(y_test.mean() * 100, 2))

## 8. Build Multiple ML Models

**Segment 8.1 – Define three models.** All use `class_weight="balanced"` so mistakes on the rare *churn* class are penalised more (handles imbalance).
- **Logistic Regression** – simple linear baseline, easy to explain.
- **Decision Tree** – rule-based; `max_depth=5` stops it memorising the data.
- **Random Forest** – many trees voting; usually strongest on tabular data.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1),
}

pipelines = {name: build_pipeline(selected_num, selected_cat, model) for name, model in models.items()}

**Segment 8.2 – Train every pipeline on the training set** (the pipeline does feature engineering → scaling/encoding → model in one `fit`).

In [ ]:
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    print("trained:", name)

## 9. Model Evaluation

**Why accuracy alone is not enough:** with 85.6% "no churn", a dummy model that always predicts "no" gets ~85.6% accuracy but catches **0** churners. We prove it below.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
d_pred = dummy.predict(X_test)
print("Dummy accuracy:", round(accuracy_score(y_test, d_pred), 4))
print("Dummy recall  :", round(recall_score(y_test, d_pred), 4), "(it catches no churners at all)")

**Segment 9.1 – Metrics for every model.**
- **Precision** = of those predicted churn, how many really churned.
- **Recall** = of all real churners, how many we caught (most important for churn: missing a churner costs money).
- **F1** = balance of precision and recall.
- **ROC-AUC** = how well the model ranks churners above non-churners (threshold independent).

In [ ]:
def evaluate(pipe, X_data, y_data):
    pred = pipe.predict(X_data)
    prob = pipe.predict_proba(X_data)[:, 1]
    return {
        "Accuracy": accuracy_score(y_data, pred),
        "Precision": precision_score(y_data, pred),
        "Recall": recall_score(y_data, pred),
        "F1": f1_score(y_data, pred),
        "ROC-AUC": roc_auc_score(y_data, prob),
    }

results = pd.DataFrame({name: evaluate(p, X_test, y_test) for name, p in pipelines.items()}).T.round(4)
results

**Segment 9.2 – Confusion matrices.** Rows = actual, columns = predicted. Bottom-left = missed churners (False Negatives), top-right = false alarms (False Positives).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (name, pipe) in zip(axes, pipelines.items()):
    ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test,
                                          display_labels=["No churn", "Churn"], cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

**Segment 9.3 – ROC curves.** The closer the curve is to the top-left corner, the better the model. The dashed line = random guessing.

In [ ]:
plt.figure(figsize=(7, 6))
for name, pipe in pipelines.items():
    fpr, tpr, _ = roc_curve(y_test, pipe.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1]):.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC curves")
plt.legend()
plt.show()

**Segment 9.4 – Train vs test comparison (overfitting check).** A big gap (train much higher than test) means the model memorised the training data.

In [ ]:
train_vs_test = pd.DataFrame({
    name: {"Train F1": f1_score(y_train, p.predict(X_train)),
           "Test F1": f1_score(y_test, p.predict(X_test)),
           "Train ROC-AUC": roc_auc_score(y_train, p.predict_proba(X_train)[:, 1]),
           "Test ROC-AUC": roc_auc_score(y_test, p.predict_proba(X_test)[:, 1])}
    for name, p in pipelines.items()
}).T.round(4)
train_vs_test

## 10. Hyperparameter Tuning

**Segment 10.1 – Tune Random Forest and Decision Tree with GridSearchCV.**
*Hyperparameters* are settings we choose before training (e.g. tree depth). `GridSearchCV` tries every combination using 5-fold cross-validation and keeps the best. We optimise **F1** because of the class imbalance.
Parameter names use the prefix `model__` because the model is a step called `model` inside the pipeline.

In [ ]:
rf_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [5, 8, 12, None],
    "model__min_samples_leaf": [1, 3, 5],
}

rf_search = GridSearchCV(
    build_pipeline(selected_num, selected_cat,
                   RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    param_grid=rf_grid, scoring="f1", cv=cv, n_jobs=-1,
)
rf_search.fit(X_train, y_train)
print("Best RF params:", rf_search.best_params_)
print("Best CV F1    :", round(rf_search.best_score_, 4))

In [ ]:
dt_grid = {
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__min_samples_leaf": [1, 5, 10, 20],
}

dt_search = GridSearchCV(
    build_pipeline(selected_num, selected_cat,
                   DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    param_grid=dt_grid, scoring="f1", cv=cv, n_jobs=-1,
)
dt_search.fit(X_train, y_train)
print("Best DT params:", dt_search.best_params_)
print("Best CV F1    :", round(dt_search.best_score_, 4))

**Segment 10.2 – Before vs after tuning (on the untouched test set).**

In [ ]:
tuned = {"Random Forest (tuned)": rf_search.best_estimator_,
         "Decision Tree (tuned)": dt_search.best_estimator_}

before_after = pd.concat([
    results.loc[["Random Forest", "Decision Tree"]],
    pd.DataFrame({n: evaluate(p, X_test, y_test) for n, p in tuned.items()}).T.round(4),
])
before_after

## 11. Final Model Selection

**Segment 11.1 – Compare all candidate models on every criterion**, including **inference time** (how long to predict) and a rough **complexity** indicator. The highest accuracy is *not* automatically the winner.

In [ ]:
candidates = {
    "Logistic Regression": pipelines["Logistic Regression"],
    "Decision Tree (tuned)": tuned["Decision Tree (tuned)"],
    "Random Forest": pipelines["Random Forest"],
    "Random Forest (tuned)": tuned["Random Forest (tuned)"],
}

rows = {}
for name, pipe in candidates.items():
    metrics = evaluate(pipe, X_test, y_test)
    start = time.perf_counter()
    for _ in range(20):
        pipe.predict(X_test)
    metrics["Inference ms / 632 rows"] = (time.perf_counter() - start) / 20 * 1000
    rows[name] = metrics

final_table = pd.DataFrame(rows).T.round(4)
final_table

**Segment 11.2 – Decision: Random Forest (tuned).**

| Criterion | What the table shows |
|---|---|
| Recall | 0.879 – catches ~88% of churners (untuned RF: 0.846, Logistic Regression: 0.868) |
| ROC-AUC | 0.9605 – best of all models |
| Overfitting | Untuned RF memorised the training set (train AUC = 1.0). Tuning (`max_depth=8`, `min_samples_leaf=3`) limits this |
| F1 / Accuracy | Slightly lower than the untuned RF (0.784 vs 0.819) – the price of catching more churners (lower precision) |
| Inference time | ~0.17 ms per row – more than the linear model but far below anything a user would notice |
| Complexity | Higher than Logistic Regression, but not needed to be explained to the end user |

**Why not the highest accuracy / F1?** The untuned Random Forest has the top accuracy, but it is overfit and misses more churners. For churn, a *missed churner* (False Negative) costs more than a *false alarm* (False Positive: we just offer a discount to someone who would have stayed), so we favour **Recall and ROC-AUC**.

**Honest caveat:** the test set has only ~91 churners, so a 3-point difference is about 3 customers. The tuned model's gains are small and should be confirmed with more data.

In [ ]:
# Choose the final model here (change the name if your table says otherwise)
final_name = "Random Forest (tuned)"
final_model = candidates[final_name]
print("Selected:", final_name)
print(final_table.loc[final_name])

## 12. Save the Model

**Segment 12.1 – Save with joblib.** We save the *whole pipeline* (feature engineering + scaling/encoding + model) in one file, so the API only has to call `predict_proba` on raw input.
We also save a small JSON with version, metrics and column info (model versioning).

In [ ]:
import os
os.makedirs("model", exist_ok=True)

MODEL_VERSION = "1.0.0"
joblib.dump(final_model, f"model/churn_model_v{MODEL_VERSION}.joblib")

metadata = {
    "model_name": final_name,
    "model_version": MODEL_VERSION,
    "target": "Churned",
    "numeric_features": selected_num,
    "categorical_features": selected_cat,
    "metrics_on_test": {k: round(float(v), 4) for k, v in final_table.loc[final_name].items()},
}
with open("model/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

**Segment 12.2 – Reload test.** Always prove the saved file works by loading it fresh and predicting on one raw row.

In [ ]:
loaded = joblib.load(f"model/churn_model_v{MODEL_VERSION}.joblib")
sample = X_test.iloc[[0]]
print(sample.T)
print("Predicted class      :", loaded.predict(sample)[0])
print("Churn probability    :", round(loaded.predict_proba(sample)[0, 1], 4))
print("Actual               :", y_test.iloc[0])